In [4]:
import pickle
from pathlib import Path

import pandas as pd
from tensorflow.keras.models import load_model

base_dir = Path.cwd()

model_path = base_dir / "model.h5"
model = load_model(model_path)

# Load encoders and scalers
# The saved file is named "lable_encoder_gender.pkl" (with a typo in "label")
possible_gender_files = [
    base_dir / "lable_encoder_gender.pkl",
    base_dir / "label_encoder_gender.pkl",
]

gender_file = next((p for p in possible_gender_files if p.exists()), None)
if gender_file is None:
    raise FileNotFoundError("Could not find the gender encoder pickle file.")
with open(gender_file, "rb") as file:
    label_encoder_gender = pickle.load(file)

with open(base_dir / "onehot_encoder_geo.pkl", "rb") as file:
    onehot_encoder_geo = pickle.load(file)

with open(base_dir / "scaler.pkl", "rb") as file:
    scaler = pickle.load(file)

# Example data
input_data = {
    "CreditScore": 600,
    "Geography": "France",
    "Gender": "Male",
    "Age": 40,
    "Tenure": 3,
    "Balance": 60000,
    "NumOfProducts": 2,
    "HasCrCard": 1,
    "IsActiveMember": 1,
    "EstimatedSalary": 50000,
}

# Encode gender
input_data["Gender"] = label_encoder_gender.transform([input_data["Gender"]])[0]

# One-hot encode geography
geo_encoded = onehot_encoder_geo.transform([[input_data["Geography"]]]).toarray()
columns_names = onehot_encoder_geo.get_feature_names_out()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=columns_names)

# Prepare final feature dataframe
input_data.pop("Geography")
input_df = pd.DataFrame([input_data])
final_df = pd.concat([input_df, geo_encoded_df], axis=1)

# Scale features
input_scaled = scaler.transform(final_df)

# Model prediction
prediction = model.predict(input_scaled)
prediction_probab = float(prediction[0][0])
print(prediction_probab)

if prediction_probab > 0.5:
    print("Employee will not leave")
else:
    print("Employee will leave")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
0.019152211025357246
Employee will leave


d:\GenAI2026\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [ ]:
## Load the trained model , scaler pickle , onehot endocer
